# ⚡ StreamVault Cloud Leech Worker (Google Colab Edition)

This notebook downloads torrents at **50 to 200 MB/s** on Google Cloud data center fiber and uploads them directly into your private Telegram Vault.

* **Your Mobile Hotspot Data Used:** **0 MB** (Google Cloud handles all network traffic)
* **Auto-Split:** Files exceeding 1.95 GB are automatically split losslessly with `ffmpeg -c copy`.
* **Queue-Driven:** Automatically processes all download requests queued from `@perso_streamvault_bot`.


### 1. Install System & Python Dependencies
Run this cell once when starting the Colab session.

In [ ]:
!apt-get update -qq && apt-get install -y -qq aria2 ffmpeg
!pip install -q hydrogram tgcrypto cryptg motor certifi requests

### 2. Configure Your StreamVault Credentials
Enter the credentials from your `.env` file below.

In [ ]:
API_ID = 0  #@param {type:"integer"}
API_HASH = ""  #@param {type:"string"}
BOT_TOKEN = ""  #@param {type:"string"}
LOG_CHANNEL_ID = 0  #@param {type:"integer"}
MONGO_URI = ""  #@param {type:"string"}
STREAM_URL = ""  #@param {type:"string"}

### 3. Start Queue-Driven Cloud Leech Worker
This cell continuously listens to your MongoDB Atlas queue and downloads any queued movies or web series.

In [ ]:
import os, sys, math, shutil, asyncio, subprocess, certifi
from motor.motor_asyncio import AsyncIOMotorClient
from hydrogram import Client
from bson import ObjectId

TELEGRAM_MAX_BYTES = 1950 * 1024 * 1024

def format_size(bytes_size):
    for u in ["B","KB","MB","GB","TB"]:
        if bytes_size < 1024: return f"{bytes_size:.2f} {u}"
        bytes_size /= 1024
    return f"{bytes_size:.2f} PB"

def run_aria2(magnet, out_dir="downloads"):
    os.makedirs(out_dir, exist_ok=True)
    cmd = ["aria2c", "--seed-time=0", "-x16", "-s16", "-k1M", "--dir", out_dir, magnet]
    subprocess.run(cmd, check=True)
    best, max_s = None, 0
    for root, _, files in os.walk(out_dir):
        for f in files:
            p = os.path.join(root, f)
            s = os.path.getsize(p)
            if f.lower().endswith((".mp4",".mkv",".avi",".webm",".ts",".m4v")) and s > max_s:
                max_s, best = s, p
    return best

def split_file(path):
    s = os.path.getsize(path)
    if s <= TELEGRAM_MAX_BYTES: return [path]
    try:
        d = float(subprocess.check_output(["ffprobe","-v","error","-show_entries","format=duration","-of","default=noprint_wrappers=1:nokey=1",path]).decode().strip())
    except Exception: d = 0.0
    parts = math.ceil(s / TELEGRAM_MAX_BYTES)
    if parts < 2 or d <= 0: return [path]
    seg = d / parts
    b, ext = os.path.splitext(path)
    out = f"{b}.part%02d{ext}"
    subprocess.run(["ffmpeg","-i",path,"-c","copy","-map","0","-f","segment","-segment_time",str(seg),"-reset_timestamps","1",out], check=True)
    res = [os.path.join(os.path.dirname(path) or ".", f) for f in sorted(os.listdir(os.path.dirname(path) or ".")) if f.startswith(os.path.basename(b) + ".part") and f.endswith(ext)]
    return res or [path]

async def worker_loop():
    bot = Client("colab_worker", api_id=API_ID, api_hash=API_HASH, bot_token=BOT_TOKEN, in_memory=True)
    mongo = AsyncIOMotorClient(MONGO_URI, tlsCAFile=certifi.where())
    db = mongo["stream_vault"]
    await bot.start()
    print("[+] Colab Leech Worker Online! Listening to MongoDB queue...")
    try:
        while True:
            task = await db["leech_queue"].find_one_and_update(
                {"status": "queued"},
                {"$set": {"status": "processing"}}
            )
            if not task:
                await asyncio.sleep(5)
                continue
            print(f"[*] Processing task: {task.get('title')}")
            user_id = task.get("user_id")
            try:
                if user_id:
                    await bot.send_message(user_id, f"⚡ **Google Colab Leeching Started:** `{task.get('title')}`\nDownloading at Google gigabit speed (0 MB hotspot data)...")
                out_dir = f"temp_{task['_id']}"
                vfile = run_aria2(task["magnet"], out_dir)
                pfiles = split_file(vfile)
                for i, p in enumerate(pfiles, 1):
                    lbl = f" (Part {i}/{len(pfiles)})" if len(pfiles) > 1 else ""
                    fn = os.path.basename(p)
                    fs = os.path.getsize(p)
                    msg = await bot.send_document(LOG_CHANNEL_ID, p, caption=f"🎬 {fn}{lbl}")
                    media = msg.video or msg.document
                    doc = {"file_name": fn, "file_size": fs, "file_id": media.file_id, "unique_id": media.file_unique_id, "source_chat_id": LOG_CHANNEL_ID, "mime_type": "video/mp4", "caption": f"🎬 {fn}"}
                    await db["files"].update_one({"unique_id": media.file_unique_id}, {"$set": doc}, upsert=True)
                    if user_id:
                        lnk = f"{STREAM_URL}/stream/{media.file_unique_id}" if STREAM_URL else "Saved in Vault"
                        await bot.send_cached_media(user_id, media.file_id, caption=f"✅ **Ready to Stream:** `{fn}`{lbl}\n📦 `{format_size(fs)}`\n🔗 `{lnk}`")
                await db["leech_queue"].update_one({"_id": task["_id"]}, {"$set": {"status": "completed"}})
                if os.path.exists(out_dir): shutil.rmtree(out_dir, ignore_errors=True)
                print(f"[+] Completed: {task.get('title')}")
            except Exception as e:
                print(f"[!] Failed: {e}")
                await db["leech_queue"].update_one({"_id": task["_id"]}, {"$set": {"status": "failed", "error": str(e)}})
                if user_id: await bot.send_message(user_id, f"❌ **Leech Failed:** `{e}`")
    finally:
        await bot.stop()

asyncio.run(worker_loop())